# PETER study: washing and emission analysis

Run the cells **top to bottom**. Keep this notebook and `peter_analysis.py` in the same folder.

**You choose the workbook yourself**: when you run the first code cell a file window opens. Nothing needs editing.

If you run on the simulated workbook the numbers only test the code — they are not findings.

| Cells | Answers |
|-------|---------|
| 1–2 | Load data, preview |
| 3–4 | **Q1** Yield RSM + diagnostics + equation |
| 5–7 | **Q1 cont.** Split-plot emission models, BH correction |
| 8–9 | **Q2** Operating window + contour plots |
| 10  | **Q3** Carbon dose–response |

In [ ]:
# 1. Setup and choose your workbook
import os, sys, re, time, glob, pathlib, importlib.util
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

script = pathlib.Path(os.getcwd()) / 'peter_analysis.py'
if not script.exists():
    raise FileNotFoundError(f'peter_analysis.py not found in {os.getcwd()}')
text = script.read_text(encoding='utf-8', errors='ignore')
m = re.search(r'__version__\s*=\s*"([^"]+)"', text)
found = m.group(1) if m else 'none (old script)'
print(f'Script : {script}')
print(f'Size   : {script.stat().st_size:,} bytes  |  saved {time.ctime(script.stat().st_mtime)}')
print(f'Version: {found}')
if not m or tuple(int(x) for x in re.findall(r'\d+', found)) < (3, 0):
    raise RuntimeError(
        'peter_analysis.py is outdated (version < 3.0).\n'
        f'File: {script}\n'
        'Replace its content with the latest peter_analysis.py, SAVE (Ctrl+S), '
        'then Kernel > Restart & Run All.')

sys.modules.pop('peter_analysis', None)
spec = importlib.util.spec_from_file_location('peter_analysis', script)
pa   = importlib.util.module_from_spec(spec)
sys.modules['peter_analysis'] = pa
spec.loader.exec_module(pa)

def choose_workbook():
    try:
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk(); root.withdraw(); root.attributes('-topmost', True)
        path = filedialog.askopenfilename(
            title='Choose the PETER workbook',
            filetypes=[('Excel workbook', '*.xlsx *.xlsm')])
        root.destroy()
        if path: return path
    except Exception:
        pass
    return input('Type or paste the full path to your workbook (.xlsx): ').strip().strip('"')

WORKBOOK = None
if WORKBOOK is None:
    WORKBOOK = choose_workbook()
pa.check_workbook(WORKBOOK)
OUT = 'results'
os.makedirs(OUT, exist_ok=True)
pd.set_option('display.width', 140)
print('\nUsing:', WORKBOOK)

In [ ]:
# 2. Load and preview
wash = pa.load_washing(WORKBOOK)
runs = pa.load_runs(WORKBOOK)
print(f'Washing rows : {len(wash)}  ({wash["Md"].notna().sum()} have Md)')
print(f'PETER_runs   : {len(runs)} rows from {runs["wash"].nunique()} washes '
      f'x {runs["Cpct"].nunique()} carbon doses')
print('\nWashing (first 5):')
display(wash[["Run","T","t","W","Md","SRY"]].head())
print('\nMissing values in model columns:')
mv_cols = ["Md"] + [c for c in runs.columns if c.endswith(("net_peak","net_plateau"))]
display(pd.concat([
    wash[["Md"]].isna().sum().rename("Washing"),
    runs[[c for c in mv_cols[1:] if c in runs]].isna().sum().rename("PETER_runs")
], axis=1).fillna(""))

## Q1 — Part A: Yield model (Md)

Fits a **second-order response surface** for final dry mass. Reports R², adjusted R², **predicted R² (PRESS)**, Box-Cox transform check, ANOVA, lack of fit, residual diagnostics, reduced model and equation.

**Predicted R²** is the honest number — it shows how well the model predicts a wash it has never seen.

In [ ]:
# 3. Part 1: yield RSM
wash_model = pa.part1(wash, OUT)

In [ ]:
# 4. Part 1 plots and equation
display(Image(os.path.join(OUT, 'part1_fit.png')))
display(Image(os.path.join(OUT, 'diag_part1.png')))
eq_path = os.path.join(OUT, 'equation_Md.txt')
if os.path.exists(eq_path):
    print('\n--- Model equation (coded variables) ---')
    print(open(eq_path).read())

## Q1 — Part B: Emission analysis (split-plot)

| Step | What it does |
|------|--------------|
| **(a) Mixed model** | Full dataset; wash = random intercept; estimates carbon and washing effects jointly |
| **(b) Whole-plot F-test** | RSM on 27 wash *means* — the **exact** test for temperature, time and water ratio |
| **(c) Tobit** | Only when >=5% of readings hit the sensor ceiling; cluster-robust SEs |

**ICC** = fraction of total variance that is wash-to-wash.

In [ ]:
# 5. Part 2: emissions
results = pa.part2(runs, OUT, wash_model)

In [ ]:
# 6. Emission residual plots
for f in sorted(glob.glob(os.path.join(OUT, 'diag_*_net_peak.png'))):
    gas = os.path.basename(f).replace('diag_','').replace('_net_peak.png','')
    print(f'\n{gas}')
    display(Image(f))

In [ ]:
# 7. BH correction + reduced model equations
bh_path = os.path.join(OUT, 'bh_correction.csv')
if os.path.exists(bh_path):
    print('Multiple-testing correction (Benjamini-Hochberg FDR):')
    print('Controls the false-discovery rate across all gas x kind model F-tests.\n')
    display(pd.read_csv(bh_path, index_col=0).round(4))
else:
    print('BH table not yet written (run Part 2 first).')

print('\n--- Reduced whole-plot equations ---')
for f in sorted(glob.glob(os.path.join(OUT, 'equation_*_net_peak.txt'))):
    gas = os.path.basename(f).replace('equation_','').replace('_net_peak.txt','')
    print(f'\n{gas}:')
    print(open(f).read())

## Q2 — Operating window

Find washing conditions that **simultaneously** keep yield >= target and cut all four gas emissions by at least the required fraction.

Change the three variables below and rerun:
- `DOSE` — carbon level to build the window for  
- `YIELD_MIN` — minimum acceptable recovery (%)  
- `REDUCTION` — minimum emission cut vs reference (0.5 = 50%)  

**Green = robust** (passes even with 95% confidence bounds). **Amber = point only**. **Grey = fails.**

**Desirability D** = 0–1 geometric mean of yield and emission targets. Best D inside the feasible region is the **desirability optimum**.

In [ ]:
# 8. Operating window
DOSE       = 10       # carbon dose (%): 0, 5, 10 or 15
YIELD_MIN  = 80.0     # minimum recovery (%)
REDUCTION  = 0.50     # required emission cut (0.50 = 50%)

ctrl = pa.load_controls(WORKBOOK)
grid = pa.window(wash_model, runs, OUT, controls=ctrl,
                 dose=DOSE, reduction=REDUCTION, yield_min=YIELD_MIN)

In [ ]:
# 9. Window plots and feasible-settings table
if grid is None:
    print('No window was calculated (see the message above).')
else:
    display(Image(os.path.join(OUT, 'window_by_ratio.png')))
    contour = os.path.join(OUT, 'window_contour.png')
    if os.path.exists(contour):
        print('\nDesirability contour (W = 10 mL/g):')
        display(Image(contour))
    for f in sorted(glob.glob(os.path.join(OUT, 'contour_*.png'))):
        gas = os.path.basename(f).replace('contour_','').replace('.png','')
        print(f'\n{gas} relative emission:')
        display(Image(f))

    ok = grid[grid.feasible_robust] if grid.feasible_robust.any() else grid[grid.feasible_point]
    if ok.empty:
        print(f'\nNo setting meets every target (yield >= {YIELD_MIN}% '
              f'and emission cut >= {REDUCTION:.0%}).\n'
              'Try lowering REDUCTION (e.g. 0.25) or YIELD_MIN above and rerun.')
    else:
        rel_cols = [c for c in ok.columns if c.endswith('_rel') and not c.endswith('_rel_hi')]
        show = ok[['T','t','W','SRY','desirability'] + rel_cols].round(3)
        print(f'\nFeasible settings (top 15 by desirability D):')
        display(show.sort_values('desirability', ascending=False).head(15))

## Q3 — Carbon dose effect

Does activated carbon reduce emissions further?

Change `GAS` to `HCHO`, `TVOC`, `CO` or `CO2`.

In [ ]:
# 10. Carbon dose-response
GAS = 'HCHO'    # change to TVOC, CO or CO2

y_pk = f'{GAS}_net_peak'
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
if y_pk in runs and runs[y_pk].notna().sum() > 0:
    means = runs.groupby('Cpct')[y_pk].mean()
    axes[0].bar(means.index.astype(str), means.values, color='#2a9d8f', width=0.5)
    axes[0].set(xlabel='Carbon dose (%)', ylabel=f'{GAS} net peak (mean)',
                title='Mean net peak by dose')
    for T, g in runs.groupby('T'):
        m = g.groupby('Cpct')[y_pk].mean()
        axes[1].plot(m.index, m.values, marker='o', label=f'{T} C')
    axes[1].set(xlabel='Carbon dose (%)', ylabel=f'{GAS} net peak',
                title='Dose x temperature')
    axes[1].legend(fontsize=8)
else:
    for ax in axes: ax.set_visible(False)
    print(f'No {y_pk} data yet.')
plt.tight_layout(); plt.show()

print('\nAll gases — net peak by carbon dose:')
fig2, ax2 = plt.subplots(2, 2, figsize=(12, 7))
for ax, g in zip(ax2.flat, pa.GASES):
    yg = f'{g}_net_peak'
    if yg not in runs or runs[yg].isna().all():
        ax.set_visible(False); continue
    runs.boxplot(column=yg, by='Cpct', ax=ax)
    ax.set(title=g, xlabel='Carbon dose (%)', ylabel='Net peak')
    plt.sca(ax); plt.title(g)
plt.suptitle('Net peak by carbon dose', y=1.01)
plt.tight_layout(); plt.show()

## Notes

- Yield is on an **as-weighed** basis (moisture columns blank in current workbook).
- **PM2.5 and PM10 are not modelled** — sensor at ceiling throughout.
- **CO2** peaks are right-censored at 5000 ppm; Tobit regression applied automatically.
- The operating window is a model prediction — verify with a **confirmation run** in the lab.
- Results from the **simulated workbook** only test the code and are not project findings.